In [ ]:
# 1D Signal Testing
from signal_1d import hurst_rs
from signal_1d import box_count_dimension
import numpy as np
import matplotlib.pyplot as plt

if __name__ == "__main__":
    Ns = [256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1048576, 2097152] #, 4194304, 8388608, 16777216]

    boxerrs = []
    rserrs = []
    avgerrs = []

    for N in Ns:
        signal = np.random.normal(0, 10, N) # White noise signal
        motion = np.cumsum(signal)

        boxdim, log_eps_inv, log_counts = box_count_dimension(motion)
        hurstrs = hurst_rs(signal)
        boxerr = 1.5 - boxdim
        rserr = 0.5 - hurstrs

        boxerrs.append(boxerr)
        rserrs.append(rserr)
        avgerrs.append((boxerr + rserr)/2)

        print(f"Error with box-counting for {N} points: {boxerr}")
        print(f"Error with R/S analysis for {N} points: {rserr}")
    
    plt.plot(Ns, boxerrs, label='boxerrs', marker='o', linestyle='-', color='blue')
    plt.plot(Ns, rserrs, label='rserrs', marker='s', linestyle='--', color='red')
    plt.plot(Ns, avgerrs, label='avgerrs', marker='x', color='green')

    # Add labels, title, and a legend
    plt.xlabel('Ns')
    plt.ylabel('Errors')
    plt.title('Errors vs. Ns')
    plt.legend()

    # Optional: Add a grid for better readability
    plt.grid(True)

    # Show the plot
    plt.show()

In [ ]:
# 2D Array Testing
from noise import pnoise2
import numpy as np
from noise_2d import power_spectrum
from noise_2d import variogram
from perlin_gen import generate_perlin_2d
from perlin_gen import array_to_image
import matplotlib.pyplot as plt
from tqdm import tqdm
# from perlin_gen import array_to_custom_color_image
# from perlin_gen import array_to_purple_black_image

shape = (512, 512)

# single_octave = generate_perlin_2d(shape, scale=50.0, octaves=16)
# array_to_image(single_octave, "perlin_16_octave.png")

n = 67

if __name__ == "__main__":    
    param = input("Test parameter (p = persistence, s = scale, o = octaves, im = image generation, c = convergence rate): ")
    num_samples = 30
    x_vals = []
    psvals = []
    vvals = []

    if param == 'p':
        pvals = np.linspace(0, 1, 11)
        x_vals = pvals
        for p in tqdm(pvals, desc="Varying Persistence"):
            ps_temp = []
            v_temp = []
            
            # Inner loop: Generate 10 samples and collect their estimates
            for _ in range(num_samples):
                im = generate_perlin_2d(shape, scale=50, octaves=6, persistence=p, seed=int(p+3))
                ps_temp.append(power_spectrum(im))
                v_temp.append(variogram(im))
                
            # Average the 10 samples and append to the main lists
            psvals.append(np.mean(ps_temp))
            vvals.append(np.mean(v_temp))
    elif param == 's':
        svals = np.linspace(10, 100, 10)
        x_vals = svals
        print(f"\n--- Estimating FD (Scale), {num_samples} samples per value ---")
        
        for s in tqdm(svals, desc="Varying Scale"):
            ps_temp = []
            v_temp = []
            for _ in range(num_samples):
                im = generate_perlin_2d(shape, scale=s, octaves=6, persistence=0.5, seed=int(s))
                ps_temp.append(power_spectrum(im))
                v_temp.append(variogram(im))
                
            psvals.append(np.mean(ps_temp))
            vvals.append(np.mean(v_temp))
    elif param == 'o':
        ovals = list(range(6, 16))
        x_vals = ovals
        print(f"\n--- Estimating FD (Octaves), {num_samples} samples per value ---")
        
        for o in tqdm(ovals, desc="Varying Octaves"):
            ps_temp = []
            v_temp = []
            for _ in range(num_samples):
                im = generate_perlin_2d(shape, scale=50, octaves=o, persistence=0.5, seed=n)
                ps_temp.append(power_spectrum(im))
                v_temp.append(variogram(im))
                
            psvals.append(np.mean(ps_temp))
            vvals.append(np.mean(v_temp))
    elif param == 'c':
        n = 30
        print(f"\n --- Showing Convergence Rate: 1 to {n} samples ---")
        ps_temp = []
        v_temp = []
        for _ in tqdm(range(n), desc = f"Sample size {n}"):
            im = generate_perlin_2d(shape, scale=50, octaves=6, persistence=0.707106781187, seed=100-n)
            ps_temp.append(power_spectrum(im))
            v_temp.append(variogram(im))
        print(f"Power Spectrum Value, {n} samples: {np.mean(ps_temp)}")
        print(f"Variogram Value, {n} samples: {np.mean(v_temp)}")
    
    elif param == 'im':
        single_octave = generate_perlin_2d(shape, scale=100.0, octaves=6, persistence=0.5)
        array_to_image(single_octave, "perlin_test.png")

    if len(x_vals) > 0:
        plt.figure(figsize=(10, 6))
        plt.plot(x_vals, psvals, marker='o', linestyle='-', color='b', label='Power Spectrum')
        plt.plot(x_vals, vvals, marker='s', linestyle='--', color='r', label='Variogram')
        
        # Dynamic labels based on the chosen parameter
        param_names = {'p': 'Persistence', 's': 'Scale', 'o': 'Octaves', 'c': "Sample Sizes"}
        plt.title(f'Estimated Fractal Dimension vs. Perlin Noise {param_names[param]} (Avg of {num_samples} runs)', fontsize=14)
        plt.xlabel(param_names[param], fontsize=12)
        
        plt.ylabel('Estimated Fractal Dimension (D)', fontsize=12)
        plt.grid(True, alpha=0.5)
        plt.legend(fontsize=12)
        plt.show()

In [ ]:
# Diagnose issue w/ FD

import matplotlib.pyplot as plt
import numpy as np
from noise import pnoise2
from noise_2d import power_spectrum, variogram, power_spectrum_diagnose, variogram_diagnose
from perlin_gen import generate_perlin_2d, generate_perlin_2d_non_normal
from perlin_gen import array_to_image
import matplotlib.pyplot as plt
from tqdm import tqdm

# 1. Generate ONE test image (using the parameters that failed)
# Make sure to use a large scale (e.g., scale=128) if you are doing 2048x2048!
test_image = generate_perlin_2d_non_normal((4096, 4096), scale=256, octaves=6, persistence=0.5)

# 2. Get the diagnostic data
D_ps, log_r, log_p, slope_ps, int_ps = power_spectrum_diagnose(test_image)
D_var, log_dist, log_var, slope_var, int_var = variogram_diagnose(test_image, num_samples=50000)

# 3. Create the plots
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# # --- Power Spectrum Plot ---
# plt.scatter(log_r, log_p, color='blue', alpha=0.6, label='Raw FFT Data')
# plt.plot(log_r, int_ps + slope_ps * log_r, color='red', linewidth=2, 
#          label=f'Current Fit (D = {D_ps:.2f})')
# plt.title('Power Spectrum: Log(Power) vs Log(Frequency)', fontsize=12)
# plt.xlabel('Log(Frequency r)', fontsize=10)
# plt.ylabel('Log(Power)', fontsize=10)
# plt.grid(True, linestyle='--', alpha=0.6)
# plt.legend()

# --- Variogram Plot ---
plt.scatter(log_dist, log_var, color='green', alpha=0.6, label='Raw Variogram Data')
plt.plot(log_dist, int_var + slope_var * log_dist, color='red', linewidth=2, 
         label=f'Current Fit (D = {D_var:.2f})')
plt.title('Variogram: Log(Variance) vs Log(Distance)', fontsize=12)
plt.xlabel('Log(Distance d)', fontsize=10)
plt.ylabel('Log(Variance)', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.show()